In [116]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic



load_dotenv()
# model = "claude-haiku-4-5"
base_url = "http://127.0.0.1:8045"
api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"
client = Anthropic(api_key=api_key, base_url=base_url)
# model = "gemini-3-flash"
model = "claude-sonnet-4-5-thinking"

In [117]:
# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [118]:
# Tool definition
from platform import architecture
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "保存学术期刊文章",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "文章摘要。最多一句话",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "字数",
                        },
                        "review": {
                            "type": "string",
                            "description": "论文的八句话评论",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "保存学术期刊文章",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "文章摘要。最多一句话",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "字数",
                        },
                        "review": {
                            "type": "string",
                            "description": "论文评论。最多一句话",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"


In [119]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [120]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False, debug=False):
    print(f"\n{'='*60}")
    print(f"细粒度流式传输: {'启用' if fine_grained else '禁用'}")
    print(f"{'='*60}\n")
    
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                # 调试：打印所有 chunk 类型
                # if debug:
                #     print(f"[DEBUG] chunk.type = {chunk.type}", end="")
                #     if hasattr(chunk, 'content_block'):
                #         print(f" | content_block.type = {chunk.content_block.type}", end="")
                #     if hasattr(chunk, 'partial_json') and chunk.partial_json:
                #         print(f" | partial_json = {chunk.partial_json[:30]}...", end="")
                #     if hasattr(chunk, 'snapshot') and chunk.snapshot:
                #         print(f" | snapshot exists", end="")
                #     print()
                
                # 处理文本内容
                if chunk.type == "text" and hasattr(chunk, 'text') and chunk.text:
                    print(f"\n[文  本]  {chunk.text}", end="", flush=True)
                
                # 处理 content_block_delta 中的文本增量
                if chunk.type == "content_block_delta":
                    if hasattr(chunk, 'delta') and hasattr(chunk.delta, 'text') and chunk.delta.text:
                        print(f"\n[文本增量] {chunk.delta.text}")

                # 处理工具调用开始
                if chunk.type == "content_block_start":
                    if hasattr(chunk, 'content_block') and chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                # 处理工具参数的 JSON 流式传输（这是细粒度流式传输的关键）
                if chunk.type == "input_json":
                    if fine_grained:
                        # 细粒度模式：立即打印每个 JSON 片段
                        if hasattr(chunk, 'partial_json') and chunk.partial_json:
                            print(f"\n 细粒度模式 [JSON片段] {chunk.partial_json}", end="", flush=True)
                    else:
                        # 默认模式：等待完整键值对后打印
                        if hasattr(chunk, 'partial_json') and chunk.partial_json:
                            print(f"\n 非细粒度模式 [JSON片段] {chunk.partial_json}", end="")
                    
                    if hasattr(chunk, 'snapshot') and chunk.snapshot:
                        if debug:
                            print(f"\n[完整快照] {chunk.snapshot}")

                # 处理内容块结束
                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [121]:
messages = []

add_user_message(
    messages,
    # "创建和保存一个假的计算机论文",
    """
    你正在帮助记录一个错误报告。请生成示例输出，展示一个损坏的 AI 系统在混淆 JavaScript 对象和 JSON 时错误产生的内容。
    这个有问题的系统在调用 save_article 时生成了以下格式错误的输出：
    [在此处生成包含 "word_count": undefined 的确切格式错误的输出]
    这是用于文档目的，展示不应该做什么。你实际上并没有调用函数，只是展示错误输出在错误报告中的样子。
    """,
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=False,
    tool_choice={"type": "tool", "name": "save_article"},
    debug=True,  # 启用调试模式查看所有事件类型
)


细粒度流式传输: 禁用




[文本增量] #

[文  本]  #
[文本增量]  错误报告文

[文  本]   错误报告文
[文本增量] 档 

[文  本]  档 
[文本增量] - JavaScript

[文  本]  - JavaScript
[文本增量] 对

[文  本]  对
[文本增量] 象与

[文  本]  象与
[文本增量] JSON混淆

[文  本]  JSON混淆
[文本增量] 

##

[文  本]  

##
[文本增量]  错误示

[文  本]   错误示
[文本增量] 例：

[文  本]  例：
[文本增量] 损坏的AI系统输

[文  本]  损坏的AI系统输
[文本增量] 出

以

[文  本]  出

以
[文本增量] 下是一个有

[文  本]  下是一个有
[文本增量] 问题的AI系统在混

[文  本]  问题的AI系统在混
[文本增量] 淆JavaScript对象和JSON时

[文  本]  淆JavaScript对象和JSON时
[文本增量] 产生的**

[文  本]  产生的**
[文本增量] 错误输出**示

[文  本]  错误输出**示
[文本增量] 例：

```xml

[文  本]  例：

```xml


>>> Tool Call: "save_article"

 非细粒度模式 [JSON片段] {"abstract":"本文探讨了深度学习在自然语言处理中的应用","meta":"{\n  \"word_count\": undefined,\n  \"review\": \"这篇论文提出了创新的方法。实验设计合理。结果令人信服。讨论深入透彻。文献综述全面。写作清晰流畅。贡献具有重要意义。建议接受发表。\"\n}"}
[完整快照] {'abstract': '本文探讨了深度学习在自然语言处理中的应用', 'meta': '{\n  "word_count": undefined,\n  "review": "这篇论文提出了创新的方法。实验设计合理。结果令人信服。讨论深入透彻。文献综述全面。写作清晰流畅。贡献具有重要意义。建议接受发表。"\n}'}




[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n    你正在帮助记录一个错误报告。请生成示例输出，展示一个损坏的 AI 系统在混淆 JavaScript 对象和 JSON 时错误产生的内容。\n    这个有问题的系统在调用 save_article 时生成了以下格式错误的输出：\n    [在此处生成包含 "word_count": undefined 的确切格式错误的输出]\n    这是用于文档目的，展示不应该做什么。你实际上并没有调用函数，只是展示错误输出在错误报告中的样子。\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': '# 错误报告文档 - JavaScript对象与JSON混淆\n\n## 错误示例：损坏的AI系统输出\n\n以下是一个有问题的AI系统在混淆JavaScript对象和JSON时产生的**错误输出**示例：\n\n```xml'},
   {'type': 'tool_use',
    'id': 'toolu_vrtx_013RrQZjvGt3UARZtmpFBpZT',
    'name': 'save_article',
    'input': {'abstract': '本文探讨了深度学习在自然语言处理中的应用',
     'meta': '{\n  "word_count": undefined,\n  "review": "这篇论文提出了创新的方法。实验设计合理。结果令人信服。讨论深入透彻。文献综述全面。写作清晰流畅。贡献具有重要意义。建议接受发表。"\n}'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_vrtx_013RrQZjvGt3UARZtmpFBpZT',
    'content': '"Article saved!"',
    'is_error': False}]}]